# Data Validation

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 15.01.2025
- Date of Last Modification: 15.01.2025 (Sebastian Birk; sebastian.birk@helmholtz-munich.de)

## 1. Setup

### 1.1 Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyensembl
#import s3fs
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
import tarfile
#import zarr
from IPython.display import HTML, display
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

import terra

INFO:datasets:PyTorch version 2.4.1 available.
INFO:datasets:Polars version 0.20.31 available.
INFO:datasets:JAX version 0.4.33 available.


### 1.2 Run Notebook Setup

In [3]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
sc.settings.set_figure_params(
    dpi=100,
    dpi_save=100,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

### 1.3 Configure Paths and Create Directories

In [5]:
# Define paths
st_data_folder_path = "/lustre/scratch126/cellgen/team361/DATASETS" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
st_data_url_folder_path = f"{st_data_folder_path}/urls"

artifacts_folder_path = '../../artifacts'
data_artifacts_folder_path = '../../artifacts/data'

# Create required directories
os.makedirs(st_data_bronze_folder_path, exist_ok=True)
os.makedirs(st_data_silver_folder_path, exist_ok=True)
os.makedirs(st_data_gold_folder_path, exist_ok=True)
os.makedirs(st_data_url_folder_path, exist_ok=True)

os.makedirs(data_artifacts_folder_path, exist_ok=True)

## 1.4 Define Functions

In [6]:
def get_directory_names(folder_path):
    try:
        # List all entries in the folder
        entries = os.listdir(folder_path)
        
        # Filter out entries that are directories
        directories = [entry for entry in entries if os.path.isdir(os.path.join(folder_path, entry))]
        
        return directories
    except FileNotFoundError:
        print(f"The folder path '{folder_path}' does not exist.")
        return []
    except PermissionError:
        print(f"Permission denied to access '{folder_path}'.")
        return []

# 2. Data Validation

In [ ]:
corpus_dataset_list = [
    'xhb23-1b_1p',
    'xhb24-2b_1p',
    'xhb25-2b_1p',
    'xhb26-2b_1p',
    'xhb27-1b_1p',
    'xhl28-1b_1p',
    'xhl29-1b_1p',
    'xhp30-1b_1p',
    'xhs31-2b_1p',
    'xhh32-1b_1p',
    'xhln33-1b_1p',
    'xhp34-1b_1p',
    'xhp35-1b_1p',
    'xhs36-2b_1p',
    'xhb37-3b_1p',
    'xhc38-4b_1p',
    'xhk39-2b_1p',
    'xhl40-2b_2p',
    'xhl41-2b_1p',
    'xhb42-3b_1p',
    'xht43-2b_1p',
    'xhsc44-6b_1p',
    'xhc45-1b_1p',
    'xhl46-1b_1p',
    'xhp48-1b_1p',
    'xhs49-1b_1p',
    'xho50-1b_1p',
    'xhc51-1b_1p',
    'xhd52-1b_1p',
    'xhl53-1b_1p',
    'xho54-1b_1p',
    'xhc55-3b_1p',
    'xhs62-1b_1p',
    'xhb76-16b_1p',
    'xhk79-4b_1p',
    'xhs1000-39b_1p',
    'xhb1002-AT3-1b_1p',
    'xhb1002-AT4-3b_1p',
    'xhb1002-AT5-1b_1p',
    'xhb1002-AT9-6b_1p',
    'xhb1002-AT10-7b_1p',
    'xhb1002-AT12-1b_1p',
    'xhb1002-AT13-2b_1p',
    'xhb1002-AT14-1b_1p',
    'xhb1002-AT15-10b_1p',
    'xhb1002-AT16-15b_1p',
    'xhb1002-AT17-2b_1p',
    'xhb1002-AT18-2b_1p',
    'xhb1002-AT19-1b_1p',
    'xhb1002-AT20-2b_1p',
    'xhb1002-AT21-2b_1p',
    'xhb1002-AT23-9b_1p',
    'xhb1002-AT101-6b_1p',
    'xhb1003-1b_1p',
    'xhln1004-1b_1p',
    'xhl1005-1b_1p',
    'mhb14-1b_1p',
    'mhc15-2b_1p',
    'mhl16-2b_1p',
    'mhl17-2b_1p',
    'mhm18-2b_1p',
    'mho19-4b_1p',
    'mhp20-2b_1p',
    'mhu21-3b_1p',
    'mhl63-10b_1p',
    'chb56-1b_1p',
    'chl57-2b_1p',
    'chl59-8b_1p',
    'chc61-9b_1p',
    'chr78-11b_1p']

corpus_dataset_list.sort()
print(len(corpus_dataset_list))

In [ ]:
silver_dataset_list = get_directory_names(st_data_silver_folder_path)
silver_dataset_list.sort()
silver_only_dataset_list = [dataset for dataset in silver_dataset_list if dataset not in corpus_dataset_list]
print(silver_only_dataset_list)

In [ ]:
bronze_dataset_list = get_directory_names(st_data_bronze_folder_path)
bronze_dataset_list.sort()
bronze_only_dataset_list = [dataset for dataset in bronze_dataset_list if dataset not in corpus_dataset_list]
print(bronze_only_dataset_list)

In [ ]:
columns = [
    'dataset',
    'assay',
    'tissue',
    'file_path',
    'batch',
    'n_cells',
    'n_genes',
    'min_counts_per_cell',
    'max_counts_per_cell',
    'median_counts_per_cell',
    'min_genes_per_cell',
    'max_genes_per_cell',
    'median_genes_per_cell'
]

df = pd.DataFrame(columns=columns)

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}'):
    for file_idx, file in enumerate(files):
        if (subdir.split('/')[-1][1] == 'h') & (file.endswith('.h5ad')): # human adata
            print(f"Dataset {subdir.split('/')[-1]}")
            print(f"File {file}")
            
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            print(adata.uns['assay'])
            print(adata.uns['tissue'])
            print(adata.uns['batch'])

            # Compute QC metrics if not available
            if ("total_counts" not in adata.obs) or ("n_genes_by_counts" not in adata.obs):
                sc.pp.calculate_qc_metrics(adata, inplace=True, percent_top=None)

            print("====================")
            print(f"Number of cells: {len(adata)}.")
            print(f"Number of genes: {len(adata.var_names)}.")
            print(f"Minimum counts per cell: {np.min(np.array(adata.X.sum(1)).flatten())}.")
            print(f"Maximum counts per cell: {np.max(np.array(adata.X.sum(1)).flatten())}.")
            print(f"Median counts per cell: {np.median(np.array(adata.X.sum(1)).flatten())}.")
            print(f"Minimum genes per cell: {min(adata.obs['n_genes_by_counts'])}.")
            print(f"Maximum genes per cell: {max(adata.obs['n_genes_by_counts'])}.")
            print(f"Median genes per cell: {np.median(adata.obs['n_genes_by_counts'])}.")

            row = {
                   'dataset': subdir.split('/')[-1],
                   'assay': adata.uns['assay'],
                   'tissue': adata.uns['tissue'],
                   'file_path': file_path,
                   'batch': adata.uns['batch'],
                   'n_cells': len(adata),
                   'n_genes': len(adata.var_names),
                   'min_counts_per_cell': np.min(np.array(adata.X.sum(1)).flatten()),
                   'max_counts_per_cell': np.max(np.array(adata.X.sum(1)).flatten()),
                   'median_counts_per_cell': np.median(np.array(adata.X.sum(1)).flatten()),
                   'min_genes_per_cell': min(adata.obs['n_genes_by_counts']),
                   'max_genes_per_cell': max(adata.obs['n_genes_by_counts']),
                   'median_genes_per_cell': np.median(adata.obs['n_genes_by_counts'])
               }
            
            df = df.append(row, ignore_index=True)
            df.to_csv(f'{data_artifacts_folder_path}/corpus_summary.csv', index=False)
                        
            sns.jointplot(data=adata.obs,
                          x="total_counts",
                          y="n_genes_by_counts",
                          kind="hex")

            if 'spatial' in adata.uns.keys():
                del(adata.uns['spatial'])
            
            sc.pl.spatial(adata,
                          color="n_genes_by_counts",
                          spot_size=40)

In [33]:
adata.obs.head()

,cell_id,transcript_counts,control_probe_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,n_genes,original_cell_id,n_genes_by_counts,log1p_n_genes_by_counts,log1p_total_counts
aaabbkjn-1,39_batch1_0,213,0,0,0,0,213,272.337354,19.462344,49,aaabbkjn-1,49,3.912023,5.365976
aaabgihg-1,39_batch1_1,152,0,0,0,0,152,117.677192,19.642969,47,aaabgihg-1,47,3.871201,5.030438
aaabmoge-1,39_batch1_2,30,0,0,0,0,30,75.681878,19.552657,22,aaabmoge-1,22,3.135494,3.433987
aaabneib-1,39_batch1_3,133,0,0,0,0,133,198.642351,43.666095,40,aaabneib-1,40,3.713572,4.897840
aaachpmg-1,39_batch1_4,87,0,0,0,0,86,132.804536,27.545313,47,aaachpmg-1,47,3.871201,4.465908


In [35]:
min(adata.obs['total_counts'])

10

In [36]:
max(adata.obs['total_counts'])

609

In [42]:
dataset_names_df = pd.read_csv(
    "/lustre/scratch126/cellgen/team361/sb75/terra-reproducibility/analysis/tokenization/human_cohort1_60m_dataset_names.txt",
    index_col=0,
    header=None)

In [49]:
len(dataset_names_df.index.tolist())

85

In [4]:
corpus_name = 'hst_corpus_110m' # 'hst_corpus_80m'
tokenized_data_str = f'{corpus_name}_32_None_None_None_None_None_shifted_log_knn_10' # 'hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10', 'hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10'
tokenized_data_folder_path = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/{corpus_name}/{tokenized_data_str}.dataset' # f'/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/{tokenized_data_str}.dataset'
silver_folder_path = '/nfs/team361/sb75/DATASETS/silver/'
tokenizer_folder_path = '/nfs/team361/sb75/DATASETS/tokenizer'

In [9]:
import os
import scanpy as sc

In [31]:
#dataset_name = 'xhp1016-18b_1p'
#dataset_name = 'xhp1017-4b_1p'
#dataset_name = 'xhk1018-23b_1p'
#dataset_name = 'xhk1019-19b_1p'
#dataset_name = 'xhl1020-16b_1p'
dataset_name = 'xhl1021-2b_1p'

In [32]:
for subdir, _, files in os.walk(silver_folder_path + dataset_name):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(len(adata))
            print(len(adata.var_names))
            print("-----")

968236
4949
-----
708121
4949
-----
